This notebook explores and analyses my YouTube watch history using Python and pandas. It covers parsing raw Google Takeout data, cleaning and structuring watch records, and performing exploratory data analysis with a focus on music listening habits

## Initiating watch history retrieval

In [4]:
from pathlib import Path
from bs4 import BeautifulSoup


project_root = Path.cwd().parent
watch_path = project_root / "data" / "raw" / "watch-history.html"

print(watch_path)
print(watch_path.exists())

C:\Users\MsnSi\PycharmProjects\Youtube_Music_Analysis\data\raw\watch-history.html
True


## Read data from watch history

Beautiful Soup is a Python library for parsing HTML/XML and turning it into something Python can navigate. Line 1 (htm) loads the html file into one string, and line 2 (soup) is used to parse through that string.


cards
│
├── cards[0] ── Watch history entry #1
├── cards[1] ── Watch history entry #2
├── cards[2] ── Watch history entry #3
├── cards[3] ── Watch history entry #4
└── ...

In [5]:
html = watch_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "lxml")

cards = soup.select("div.outer-cell")

print(f"Number of activity records: {len(cards)}")

Number of activity records: 12200


## Grabbing the data from watch history


In [6]:
first_card = cards[0]
content = first_card.select_one("div.content-cell")

list(content.stripped_strings)

['Watched',
 'Mario Kart 8 Deluxe Players are Going Insane right now...',
 'Shortcat',
 '7 Sept 2026, 13:51:45 IST']

## Converting the Watched Video Data records into a dictionary data structure

In [7]:
parts = list(content.stripped_strings)
links = content.select("a")


record = {
    "action": parts[0],
    "title": parts[1],
    "channel": parts[2],
    "watched_at": parts[3],
    "video_url": links[0].get("href"),
}

print(record)

{'action': 'Watched', 'title': 'Mario Kart 8 Deluxe Players are Going Insane right now...', 'channel': 'Shortcat', 'watched_at': '7 Sept 2026, 13:51:45 IST', 'video_url': 'https://www.youtube.com/watch?v=TOKdln0vkrw'}


In [8]:
## Creating all the Records from Watch history

Create empty records list

FOR every card:
    Find the content inside this card

    Extract the strings
    Extract the links

    Build a dictionary

    Add dictionary to records

Finished → records now contains every watch-history entry

In [15]:
records = []
record_number = 0
bad_records = 0

for card in cards:
    content = card.select_one("div.content-cell")
    parts = list(content.stripped_strings)
    links = content.select("a")

    if len(parts) < 4:
        bad_records += 1
        print("PARTS:", parts)
        print("LINKS:", links)
        print("--------------------")
    else:
        record_number += 1

    #record = {
    #    "action": parts[0],
    #    "title": parts[1],
     #   "channel": parts[2],
      #  "watched_at": parts[3],
       # "video_url": links[0].get("href"),
    #}

    #records.append(record)
print("Number of normal records", record_number)
print("Number of bad records: ", bad_records)


PARTS: ['Watched', 'INH CARDS CUSTOMCARDS 3183db07 AIAvatar FaceUGC 9x16 18s A1 NA NL NL Nov2025 VO  1', '4 Sept 2026, 21:23:10 IST']
LINKS: [<a href="https://www.youtube.com/watch?v=J5wkIKQfW_I">INH CARDS CUSTOMCARDS 3183db07 AIAvatar FaceUGC 9x16 18s A1 NA NL NL Nov2025 VO  1</a>]
--------------------
PARTS: ['Watched', 'INFNANO LIFESTYLE BUDGETING 698ef62a Ivelina2UGC UGC 9x16 NA A1 NA BG BG Jul2026 VO', '3 Sept 2026, 23:15:10 IST']
LINKS: [<a href="https://www.youtube.com/watch?v=KkFMhhuYruM">INFNANO LIFESTYLE BUDGETING 698ef62a Ivelina2UGC UGC 9x16 NA A1 NA BG BG Jul2026 VO</a>]
--------------------
PARTS: ['Viewed a post that is no longer available', '1 Sept 2026, 03:39:29 IST']
LINKS: []
--------------------
PARTS: ['Viewed a post that is no longer available', '31 Aug 2026, 14:24:55 IST']
LINKS: []
--------------------
PARTS: ['Viewed a post that is no longer available', '29 Aug 2026, 22:38:37 IST']
LINKS: []
--------------------
PARTS: ['Watched', 'https://www.youtube.com/watch

It seems like our parser is a bit flawed due to some bad data within the watch history, there seems to be 4 types of outliers:

1. Ads: 3 Parts, but no obvious link
2. Community Posts: 3 parts, no link
3. Unlisted Videos: 3 Parts, Link exists in the *links* field
4. Privated Videos: 3 Parts, 1 Link in parts, 2 links in *links field


## Updated Record Retrieval


In [24]:
records = []
record_number = 0
missing_channel_records = 0
non_video_records = 0


for card in cards:
    content = card.select_one("div.content-cell")
    parts = list(content.stripped_strings)
    links = content.select("a")

    if parts[0] != 'Watched':
        non_video_records += 1
        continue

    if len(parts) == 4:
        record_number += 1
        record = {
            "action": parts[0],
            "title": parts[1],
            "channel": parts[2],
            "watched_at": parts[3],
            "video_url": links[0].get("href"),
        }

    elif len(parts) == 3:
        missing_channel_records += 1
        record = {
            "action": parts[0],
            "title": parts[1],
            "channel": None,
            "watched_at": parts[2],
        }


    records.append(record)




print("Number of normal records", record_number)
print("Number of records with missing channels: ", missing_channel_records)
print("Number of non-video records: ", non_video_records)

Number of normal records 9945
Number of bad records with 3 parts:  115
Number of non-video records:  2140
Number of five plus part records:  0


## Implementing the Pandas Library

In [30]:
import pandas as pd

records_dataframe = pd.DataFrame(records)

records_dataframe.head()

<class 'pandas.DataFrame'>


,action,title,channel,watched_at,video_url
0,Watched,Mario Kart 8 Deluxe Players are Going Insane r...,Shortcat,"7 Sept 2026, 13:51:45 IST",https://www.youtube.com/watch?v=TOKdln0vkrw
1,Watched,Oops! Nintendo reveals Mario Kart 8's DEBUG mode,Beta64,"7 Sept 2026, 13:50:46 IST",https://www.youtube.com/watch?v=Lg1QI8ZkPRU
2,Watched,I Love These Uma Musume,Starfang,"7 Sept 2026, 13:44:34 IST",https://www.youtube.com/watch?v=5OOE4WORb5Q
3,Watched,第三方居然出了MG飛昇自由？大砲還有拉伸結構，還可以全彈髮射！ #gundam #갓건담 #...,玩具測評總動員,"7 Sept 2026, 13:43:43 IST",https://www.youtube.com/watch?v=R9922GyZmvw
4,Watched,【#なつめぇ誕生日2026】🎂ケーキ食べようよ！【来栖夏芽/にじさんじ】,来栖 夏芽-kurusu natsume-【にじさんじ】,"7 Sept 2026, 13:35:14 IST",https://www.youtube.com/watch?v=Ftji2LKqVXs
